# Logistic regression hyperparameters

In [1]:
import warnings
import numpy as np
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
warnings.simplefilter("ignore")

X, y = load_breast_cancer(return_X_y=True)
print(X.shape, np.bincount(y))

(569, 30) [212 357]


## Penalty type (scikit-learn 1.8+: l1_ratio and C instead of penalty)

In [2]:
settings = {
    "no penalty (C=inf)":           dict(C=np.inf),
    "L2, C=1 (default)":            dict(C=1.0),
    "L1, C=1":                      dict(C=1.0, l1_ratio=1, solver="saga", max_iter=20000),
    "Elastic Net 0.5, C=1":         dict(C=1.0, l1_ratio=0.5, solver="saga", max_iter=20000),
}
for name, kw in settings.items():
    m = make_pipeline(StandardScaler(), LogisticRegression(**kw))
    cv = cross_val_score(m, X, y, cv=5).mean()
    nz = (np.abs(m.fit(X, y)[-1].coef_) > 1e-8).sum()
    print(f"{name:28s} CV accuracy {cv:.3f}, non-zero coefficients {nz}")

no penalty (C=inf)           CV accuracy 0.956, non-zero coefficients 30
L2, C=1 (default)            CV accuracy 0.981, non-zero coefficients 30


L1, C=1                      CV accuracy 0.970, non-zero coefficients 16


Elastic Net 0.5, C=1         CV accuracy 0.974, non-zero coefficients 26


## C: smaller means stronger regularisation

In [3]:
for C in [0.001, 0.01, 0.1, 1, 10, 100]:
    m = make_pipeline(StandardScaler(), LogisticRegression(C=C))
    print(f"C = {C:>7}: CV accuracy {cross_val_score(m, X, y, cv=5).mean():.3f}, largest |coef| {np.abs(m.fit(X, y)[-1].coef_).max():.3f}")

C =   0.001: CV accuracy 0.891, largest |coef| 0.095
C =    0.01: CV accuracy 0.949, largest |coef| 0.259


C =     0.1: CV accuracy 0.977, largest |coef| 0.597


C =       1: CV accuracy 0.981, largest |coef| 1.321


C =      10: CV accuracy 0.968, largest |coef| 2.756
C =     100: CV accuracy 0.963, largest |coef| 7.696


## class_weight on imbalanced data
Real data: the breast-cancer tumours with malignant (1) as the rare class. All 357 benign tumours plus 20 malignant ones drawn at random (5% positives); half for training, half for testing (stratified). Averaged over 30 draws (seeds 0-29).

In [4]:
Xb, yb = load_breast_cancer(return_X_y=True)
yb = 1 - yb                                   # 1 = malignant, the rare class here
res = {None: [], "balanced": []}
for s in range(30):
    rng = np.random.default_rng(s)
    pos = rng.choice(np.where(yb == 1)[0], 20, replace=False)
    idx = np.r_[np.where(yb == 0)[0], pos]
    a, b, c, d = train_test_split(Xb[idx], yb[idx], test_size=0.5, random_state=s, stratify=yb[idx])
    for cw in res:
        p = make_pipeline(StandardScaler(), LogisticRegression(class_weight=cw)).fit(a, c).predict(b)
        res[cw].append((recall_score(d, p), precision_score(d, p, zero_division=0)))
print("test set: positives", d.sum(), "of", len(d))
for cw, v in res.items():
    r, pr = np.mean(v, axis=0)
    print(f"class_weight={cw}: recall {r:.3f}, precision {pr:.3f}")

test set: positives 10 of 189
class_weight=None: recall 0.770, precision 0.989
class_weight=balanced: recall 0.850, precision 0.872


## Solver and penalty must match

In [5]:
try:
    LogisticRegression(l1_ratio=1, solver="lbfgs").fit(X, y)
except ValueError as e:
    print("ValueError:", e)

ValueError: Solver lbfgs supports only 'l2' or None penalties, got l1 penalty.
